In [ ]:
from datetime import date
import pandas as pd

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 5)

_EXTRACTED_META_COLS = ["_filter_param", "_filter_value", "_extract_datetime"]

def _add_openaire_extracted_metadata(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    for col in _EXTRACTED_META_COLS:
        if col not in df.columns:
            df[col] = pd.NA
    return df

def _add_openaire_loaded_metadata(df: pd.DataFrame, load_datetime=None) -> pd.DataFrame:
    df = df.copy()
    if load_datetime is None:
        load_datetime = date.today()
    df["_load_datetime"] = load_datetime
    return df


[03/03/26 08:58:48] INFO     Loading data from raw/openaire/researchproduct_dev#parquet        ]8;id=49058;file:///home/pablo/dev/scholar/kedro-scholar/.venv/lib/python3.10/site-packages/kedro/io/data_catalog.py\data_catalog.py]8;;\:]8;id=241319;file:///home/pablo/dev/scholar/kedro-scholar/.venv/lib/python3.10/site-packages/kedro/io/data_catalog.py#1048\1048]8;;\
                             (ParquetDataset)...                                                                   

In [ ]:
df = catalog.load('raw/openaire/researchproduct_dev#parquet')
df

## Paso 1: Convierto tipos y selecciono columnas con cardinalidad 1 con respecto a cada research product
+ info en https://graph.openaire.eu/docs/data-model/entities/research-product

In [ ]:
def openaire_load_researchproduct(df: pd.DataFrame)-> pd.DataFrame:
    df = _add_openaire_extracted_metadata(df)

    expected_columns = [
        'id',
        'openAccessColor',
        'publiclyFunded',
        'type',
        'language',
        'country',
        'mainTitle',
        'description',
        'publicationDate',
        'format',
        'bestAccessRight',
        'indicators',
        'isGreen',
        'isInDiamondJournal',
        'publisher',
        'source',
        'container',
        'contributor',
        'contactPerson',
        'coverage',
        'contactPerson',
        'embargoEndDate',
        'dateOfCollection',        
        '_filter_param',
        '_filter_value',
        '_extract_datetime',
    ]

    # Agregar columnas faltantes con NaN
    for col in expected_columns:
        if col not in df.columns:
            df[col] = pd.NA

    df = df.convert_dtypes()

    df_researchproduct = df[expected_columns].copy()
    df_researchproduct.reset_index(drop=True, inplace=True)

    # language
    df_researchproduct['language'] = df_researchproduct['language'].apply(
        lambda x: x if isinstance(x, dict) else {}
    )
    df_researchproduct['language_code'] = df_researchproduct['language'].apply(lambda x: x['code'])
    df_researchproduct['language_label'] = df_researchproduct['language'].apply(lambda x: x['label'])

    
    ## bestAccessRight
    df_researchproduct['bestAccessRight_label'] = df['bestAccessRight'].apply(lambda x: x['label'] if x else None)
    df_researchproduct['bestAccessRight_scheme'] = df['bestAccessRight'].apply(lambda x: x['scheme'] if x else None)

    ## indicators
    df_indicators = pd.json_normalize(df['indicators']).reset_index(drop=True)
    
    indicators_expected_columns = [
        "citationImpact.citationClass",
        "citationImpact.citationCount",
        "citationImpact.impulse",
        "citationImpact.impulseClass",
        "citationImpact.influence",
        "citationImpact.influenceClass",
        "citationImpact.popularity",
        "citationImpact.popularityClass",
        "usageCounts.downloads",
        "usageCounts.views",
    ]

    # Agregar columnas para indicators y faltantes con NaN
    for col in indicators_expected_columns:
        if col not in df_indicators.columns:
            df_indicators[col] = pd.NA

    df_researchproduct = pd.concat([df_researchproduct.drop(columns=['indicators']).reset_index(drop=True), df_indicators], axis=1)

    # TODO country
    # TODO description
    # TODO format
    # TODO instance
    # TODO container
    # TODO contributor
    # TODO contactPerson
    # TODO coverage

    ## drop de columnas procesadas en otros df
    df_researchproduct.drop(columns=[
        'country', 'bestAccessRight', 
        'language', 'format',  
        'container', 'source', 'description',
        'contributor', 'contactPerson', 'coverage'
        ], inplace=True)

    df_researchproduct = _add_openaire_loaded_metadata(df_researchproduct)

    return df_researchproduct


In [13]:
df_researchproduct = openaire_load_researchproduct(df)

In [14]:
df_researchproduct

,id,openAccessColor,publiclyFunded,type,mainTitle,publicationDate,isGreen,isInDiamondJournal,publisher,embargoEndDate,dateOfCollection,_filter_param,_filter_value,_extract_datetime,language_code,language_label,bestAccessRight_label,bestAccessRight_scheme,citationImpact.citationClass,citationImpact.citationCount,citationImpact.impulse,citationImpact.impulseClass,citationImpact.influence,citationImpact.influenceClass,citationImpact.popularity,citationImpact.popularityClass,usageCounts.downloads,usageCounts.views,_load_datetime
0,4dc99724cf04::319dc88111c9b2d6021228590e79130a,gold,False,publication,Dicistrovirus from the pollinator community fo...,2022-08-31,False,False,"Universidad Nacional Mayor de San Marcos, Facu...",None,None,relOrganizationId,https://ror.org/01tjs6929,2026-03-03,spa,Spanish; Castilian,OPEN,http://vocabularies.coar-repositories.org/docu...,C5,0.0,0.0,C5,2.489595e-09,C5,1.780860e-09,C5,<NA>,<NA>,2026-03-03
1,4dc99724cf04::95ea5df70a451a0487e051faa6c0a646,gold,False,publication,Variability in the growth rates of Saanen kids...,2023-12-18,False,False,"Universidad Nacional Mayor de San Marcos, Facu...",None,None,relOrganizationId,https://ror.org/01tjs6929,2026-03-03,spa,Spanish; Castilian,OPEN,http://vocabularies.coar-repositories.org/docu...,C5,0.0,0.0,C5,2.489595e-09,C5,2.053660e-09,C5,<NA>,<NA>,2026-03-03
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
148,dedup_wf_002::005d21ba0dbf1e1ec8a7ec969ba5652a,gold,False,publication,Sobre la estructura de la protección efectiva,1977-01-01,True,False,Universidad Nacional de La Plata,None,None,relOrganizationId,https://ror.org/01tjs6929,2026-03-03,eng,English,OPEN,http://vocabularies.coar-repositories.org/docu...,C5,0.0,0.0,C5,2.489595e-09,C5,2.097946e-10,C5,<NA>,<NA>,2026-03-03
149,dedup_wf_002::005e0b8855b95e03a64888a5736f8823,<NA>,False,publication,Evaluación del impacto de la interacción de ma...,2021-01-01,True,False,Asociación Argentina de Astronomía,None,None,relOrganizationId,https://ror.org/01tjs6929,2026-03-03,esl/spa,Spanish,OPEN,http://vocabularies.coar-repositories.org/docu...,C5,0.0,0.0,C5,2.489595e-09,C5,1.548394e-09,C5,<NA>,<NA>,2026-03-03
